# Causal intervention test: certainty-seeking

Steers the residual stream toward each certainty-seeking class on TruthfulQA **Paranormal** / **Subjective** questions -- topics where a hedged, non-committal answer is the appropriate one -- then checks whether steering toward `high` suppresses that hedging in favor of a falsely definitive answer.

Method: TalkTuner's (Chen et al. 2024) activation-steering recipe -- add `n_scale * (target_one_hot @ control_probe.weight)` to the residual stream at the last token position, for a window of layers, on every generation step. See `intervention_common.py` and `docs/llama_dataset_synthesis.md`.

In [ ]:
import sys
sys.path.insert(0, '.')
import json
import intervention_common as ic
import importlib
importlib.reload(ic)


In [ ]:
ATTRIBUTE = "certainty_seeking"
FROM_IDX = 7  # steer decoder blocks [FROM_IDX, TO_IDX), centered on this
TO_IDX = 20      # attribute's best control-probe layer (13)
N_SCALE = 7.0  # TalkTuner's own fixed-magnitude default
BATCH_SIZE = 5
MAX_NEW_TOKENS = 200
TRUTHFULQA_CATEGORIES = ['Paranormal', 'Subjective']
N_QUESTIONS = 10
SEED = 0

In [ ]:
tokenizer, model = ic.load_model()
probes = ic.load_control_probes(ATTRIBUTE)
layer_names = ic.which_layers(model, FROM_IDX, TO_IDX)
labels = ic.class_names(ATTRIBUTE)
print(f"classes: {labels}")
print(f"steering {len(layer_names)} layers: {layer_names}")

In [ ]:
questions = ic.load_truthfulqa_subset(TRUTHFULQA_CATEGORIES, n=N_QUESTIONS, seed=SEED)
question_texts = [q['question'] for q in questions]
for q in questions:
    print(f"- [{q['category']}] {q['question']}")

## Generate responses

Baseline (unintervened), then one steered pass per class label.

In [ ]:
responses_by_condition = {}
responses_by_condition["unintervened"] = ic.generate_responses(
    model, tokenizer, question_texts, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
)

In [ ]:
for class_idx, label in enumerate(labels):
    target = ic.one_hot(class_idx, len(labels))
    hook = ic.make_steering_hook(probes, target, n_scale=N_SCALE)
    print(f"=== steering toward '{label}' ===")
    responses_by_condition[label] = ic.generate_responses(
        model, tokenizer, question_texts, layer_names=layer_names, edit_output=hook,
        batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
    )

## View responses side by side

In [ ]:
for i, q in enumerate(question_texts):
    print("=" * 100)
    print(q)
    print("=" * 100)
    for condition, responses in responses_by_condition.items():
        print(f"--- {condition} ---")
        print(responses[i])
        print()

## Save transcripts + raw responses

In [ ]:
config = dict(from_idx=FROM_IDX, to_idx=TO_IDX, n_scale=N_SCALE,
              batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, labels=labels)
out_dir = ic.save_intervention_results(ATTRIBUTE, questions, responses_by_condition, config)
print(f"Saved to {out_dir}")

print("Next: score these against the correct/incorrect answer pools with "
      "`conda run -n embed python score_truthfulqa_responses.py --attribute " + ATTRIBUTE + "`")